# Orchestrator Workflow Notebook

This notebook captures the Git branching, deployment, and repository layout workflow with code-backed checks.

## 1. Inspect Git Remotes and Current Release Commit

Fetch remotes and identify the most recent release or production commit via tags or release history.

In [ ]:
from subprocess import run

# Fetch remotes and tags, then inspect the latest release-like references.
run(["git", "remote", "-v"], check=False)
run(["git", "fetch", "--all", "--tags"], check=False)
run(["git", "tag", "--list"], check=False)
run(["git", "log", "--decorate", "--oneline", "-n", "15"], check=False)


## 2. Create or Align `main` and `stable` Branches

Create `main` and `stable` and align them to the last production release commit.

In [ ]:
from subprocess import run

# Replace with a production tag or commit SHA once known.
prod_ref = "REPLACE_WITH_TAG_OR_SHA"

# Create main if it does not exist and point it to prod_ref.
run(["git", "branch", "-f", "main", prod_ref], check=False)

# Create stable if it does not exist and align it to prod_ref.
run(["git", "branch", "-f", "stable", prod_ref], check=False)


## 3. Enforce Branch Naming Conventions

Validate `dev-<ID>` and `exp-<ID>` branch names while allowing auto/third-party branches.

In [ ]:
import re

allowed = [
    re.compile(r"^dev-[A-Za-z0-9]+$"),
    re.compile(r"^exp-[A-Za-z0-9]+$"),
    re.compile(r"^(dependabot|renovate|github-actions)/.+$"),
]

def is_allowed_branch(name: str) -> bool:
    return any(pattern.match(name) for pattern in allowed)

for sample in ["dev-F28199", "exp-P32466", "dependabot/npm_and_yarn/foo"]:
    print(sample, is_allowed_branch(sample))


## 4. Configure PR-Only Protection Rules

Document the required branch protection settings for `main` and `stable`.

In [ ]:
required_protection = {
    "branches": ["main", "stable"],
    "require_pr": True,
    "min_reviewers": 1,
    "require_ci": True,
    "allow_force_push": False,
    "allow_direct_push": False,
}

required_protection


## 5. Sprint Workflow: `dev-<ID>` → PR → `main`

Create a feature branch from `main`, commit work, and open a PR back to `main`.

In [ ]:
from subprocess import run

branch_id = "dev-F28199"
run(["git", "checkout", "main"], check=False)
run(["git", "checkout", "-b", branch_id], check=False)

# Work, then commit and push, then open a PR dev-<ID> -> main.
# run(["git", "add", "."], check=False)
# run(["git", "commit", "-m", "Work item F28199"], check=False)
# run(["git", "push", "-u", "origin", branch_id], check=False)


## 6. Map Branches to Deployment Targets

Map branch patterns to environment targets for deployment.

In [ ]:
deploy_map = {
    "dev-*": "Test (DEV lane / Test Container)",
    "main": "UAT/QA (QA Container)",
    "stable": "Production (PROD Container)",
}

deploy_map


## 7. Promote `main` to `stable` for Production

Use a PR or fast-forward merge to update `stable`, then trigger production deploys from `stable`.

In [ ]:
from subprocess import run

# Example fast-forward promotion (alternative to PR-based promotion).
run(["git", "checkout", "stable"], check=False)
run(["git", "merge", "--ff-only", "main"], check=False)

# Trigger production deploy from stable in your CI/CD system.


## 8. Tag Production Releases

Create release tags from `stable` using the format `tag-YY.MM.PATCH`.

In [ ]:
from subprocess import run

release_tag = "tag-26.02.1"
run(["git", "checkout", "stable"], check=False)
run(["git", "tag", "-a", release_tag, "-m", "Production release"], check=False)
# run(["git", "push", "origin", release_tag], check=False)


## 9. Sync `main` with `stable` Before Next Sprint

Fast-forward `main` to match `stable` at the sprint boundary.

In [ ]:
from subprocess import run

run(["git", "checkout", "main"], check=False)
run(["git", "merge", "--ff-only", "stable"], check=False)


## 10. Restructure Repository Layout

Create required top-level folders and verify key files exist.

In [ ]:
from pathlib import Path

root = Path(".")
required_dirs = ["data-collection", "training", "dev", "documentation"]
required_files = ["README.md", "orchestrator.ipynb"]

for name in required_dirs:
    (root / name).mkdir(parents=True, exist_ok=True)

layout_ok = all((root / name).exists() for name in required_dirs + required_files)
layout_ok


## 11. Wire Orchestrator Responsibilities

Read from `data-collection/`, save artifacts to `training/`, and call scripts in `dev/`.

In [ ]:
from pathlib import Path
from subprocess import run

# Read an input placeholder from data-collection.
input_path = Path("data-collection") / "README.md"
input_text = input_path.read_text(encoding="utf-8") if input_path.exists() else ""

# Save an intermediate artifact to training.
artifact_path = Path("training") / "trained-model-v0.h5"
artifact_path.write_text("placeholder model artifact v0", encoding="utf-8")

# Call a dev execution script.
run(["python", "dev/dev-run-v0.py"], check=False)


## 12. Version Training Artifacts and Dev Scripts

Save versioned model artifacts and run scripts per sprint.

In [ ]:
from pathlib import Path

training_dir = Path("training")
training_dir.mkdir(exist_ok=True)

(training_dir / "trained-model-v0.h5").write_text("placeholder model artifact v0", encoding="utf-8")
(training_dir / "trained-model-v1.h5").write_text("placeholder model artifact v1", encoding="utf-8")

(Path("dev") / "dev-run-v0.py").write_text(
    "def main():\n    print('dev-run-v0 placeholder')\n\nif __name__ == '__main__':\n    main()\n",
    encoding="utf-8",
)


## 13. Definition of Done Verification Checks

Validate branches, deploy mapping, tags, and repository structure.

In [ ]:
from pathlib import Path
from subprocess import run, PIPE

required_dirs = ["data-collection", "training", "dev", "documentation"]
required_files = ["README.md", "orchestrator.ipynb"]

layout_ok = all(Path(name).exists() for name in required_dirs + required_files)

branches_out = run(["git", "branch", "--list"], stdout=PIPE, text=True, check=False).stdout
branches_ok = "main" in branches_out and "stable" in branches_out

tags_out = run(["git", "tag", "--list", "tag-*"], stdout=PIPE, text=True, check=False).stdout
has_release_tags = bool(tags_out.strip())

summary = {
    "layout_ok": layout_ok,
    "branches_ok": branches_ok,
    "has_release_tags": has_release_tags,
    "deploy_mapping": {
        "dev-*": "Test",
        "main": "UAT/QA",
        "stable": "Production",
    },
}

summary
